# OttoLatin – B1: few-shot LLM baseline (no training)

Shows Qwen2.5-Instruct 8 example pairs and asks it to transliterate Ottoman (Arabic script) → Modern Turkish (Latin). No fine-tuning.

**One-time setup (GitHub token, the repo is private):**
1. GitHub → Settings → Developer settings → Personal access tokens → **Fine-grained tokens** → Generate new token
2. Repository access: *Only select repositories* → `fatiheogan/OttoLatin`; Permissions → **Contents: Read-only** → Generate
3. In Colab, open **🔑 Secrets** (left sidebar) → *Add new secret* → Name `GITHUB_TOKEN`, Value = token → enable *Notebook access*

**Run:** Runtime → Change runtime type → **T4 GPU** → **Run all**. At the end `preds_qwen_fewshot.zip` is downloaded.
(Without a token, cell 2 asks you to upload `eval.zip`.)

- **GPU available (T4):** 1.5B model, all sets → the real B1 result (~30–60 min)
- **No GPU:** 0.5B model, 20 rows per set → pipeline test only, scores are not reported


In [ ]:
!pip -q install -U transformers accelerate
import json, os, time, torch, zipfile
from google.colab import files
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - set the runtime type to T4!")

In [ ]:
# Get the evaluation sets: from the GitHub repo first, otherwise upload eval.zip
EVAL = None
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if not os.path.exists("OttoLatin"):
        !git clone -q https://{token}@github.com/fatiheogan/OttoLatin.git
    if not os.path.isdir("OttoLatin/data/eval"):
        raise RuntimeError("clone failed (token permissions?)")
    EVAL = "OttoLatin/data/eval"
    print("Loaded from repo.")
except Exception as e:
    print("Could not load repo (", type(e).__name__, ") -> upload eval.zip")
    up = files.upload()
    with zipfile.ZipFile(next(iter(up))) as z:
        z.extractall("eval_raw")
    # zips made by Windows Compress-Archive use backslashes ("eval\\x.jsonl") -> fix
    import shutil
    os.makedirs("eval_fixed", exist_ok=True)
    for root, _, fs in os.walk("eval_raw"):
        for f in fs:
            name = f.replace("\\", "/").split("/")[-1]
            if name.endswith(".jsonl"):
                shutil.copy(os.path.join(root, f), os.path.join("eval_fixed", name))
    EVAL = "eval_fixed"
print("eval folder:", EVAL, sorted(os.listdir(EVAL)))

In [ ]:
# ---- SETTINGS ----
DEV = "cuda" if torch.cuda.is_available() else "cpu"
if DEV == "cuda":
    MODEL, LIMIT, BATCH, OUT = "Qwen/Qwen2.5-1.5B-Instruct", None, 16, "preds/qwen_fewshot"
else:   # no GPU -> pipeline test only
    MODEL, LIMIT, BATCH, OUT = "Qwen/Qwen2.5-0.5B-Instruct", 20, 4, "preds/qwen_cpu_test"
SETS = ["test_2k", "riqa", "naskh", "ot2tr"]
print(f"Device: {DEV} | Model: {MODEL} | LIMIT: {LIMIT}")
os.makedirs(OUT, exist_ok=True)

from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16 if DEV == "cuda" else torch.float32).to(DEV).eval()

load = lambda p: [json.loads(l) for l in open(p, encoding="utf-8")]
shots = load(f"{EVAL}/fewshot.jsonl")
SYSTEM = ("You transliterate Ottoman Turkish written in Arabic script into Modern Turkish Latin script. "
          "Output ONLY the transliteration on one line, nothing else.")

def build(ottoman):
    msgs = [{"role": "system", "content": SYSTEM}]
    for s in shots:
        msgs += [{"role": "user", "content": s["ottoman"]}, {"role": "assistant", "content": s["turkish"]}]
    msgs.append({"role": "user", "content": ottoman})
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

print(build("یارین قبول ایدییورم.")[-400:])

In [ ]:
@torch.no_grad()
def run(rows):
    preds = []
    for i in range(0, len(rows), BATCH):
        b = rows[i:i + BATCH]
        enc = tok([build(r["ottoman"]) for r in b], return_tensors="pt", padding=True).to(DEV)
        max_new = min(512, int(max(len(r["ottoman"]) for r in b) * 1.6) + 16)
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.eos_token_id)
        for r, o in zip(b, out[:, enc["input_ids"].shape[1]:]):
            text = tok.decode(o, skip_special_tokens=True).strip().split("\n")[0].strip()
            preds.append({"id": r["id"], "prediction": text})
        if (i // BATCH) % 10 == 0:
            print(f"  {i + len(b)}/{len(rows)}")
    return preds

for s in SETS:
    rows = load(f"{EVAL}/{s}.jsonl")[:LIMIT]
    t = time.time()
    preds = run(rows)
    with open(f"{OUT}/{s}.jsonl", "w", encoding="utf-8") as f:
        for p in preds:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")
    print(f"[{s}] {len(preds)} predictions, {time.time() - t:.0f} s")
    for r, p in list(zip(rows, preds))[:3]:
        print("   REF :", r["turkish"][:100]); print("   PRED:", p["prediction"][:100])

In [ ]:
!cd preds && zip -qr ../preds_qwen_fewshot.zip .
files.download("preds_qwen_fewshot.zip")